# Imaging configuration checks

[Open in Colab](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/imaging/demo_config_check.ipynb)

Validate imaging configurations before opening data or constructing a graph.
The examples use generated metadata and small NumPy/Xarray arrays, so no
observational data or compute cluster is required.

## Assumptions and background

Cube and continuum imaging each have single-field and multifield contexts.
Single-field execution is implemented; the multifield contexts are reserved and
raise explicitly. Configuration checks preserve caller values and never insert
defaults or substitute another algorithm. Numerical bounds and conditional
rules are checked in Python; schemas check basic structure and types.


## Install AstroVIPER

Use an existing AstroVIPER environment, or explicitly enable the optional install
cell below in a fresh Colab session. Installation is disabled by default so that
running this notebook in a development checkout does not change its environment.
The examples require the compiled extensions included with AstroVIPER.


In [ ]:
INSTALL_ASTROVIPER = False

if INSTALL_ASTROVIPER:
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--upgrade", "astroviper"]
    )

from importlib.metadata import version

print("AstroVIPER", version("astroviper"))

## API

`check_config(config, mode=..., require_complete=...)` coordinates three
independent validators: `check_residual_update`, `check_model_update`, and
`check_driver_workflow`. Drivers require a complete configuration; direct helper
calls may validate only the parameters relevant to their boundary.

### Validation through the imaging stack

1. Validate the configuration at a callable entry point.
2. A validated caller passes `input_checked=True` for the unchanged configuration
   in the same imaging mode; direct callers leave the default `False`.
3. Runtime checks on loaded data, derived products, and native buffers remain
   active regardless of that flag.


In [ ]:
import inspect

from astroviper.processing_functions.imaging.config_check import (
    check_config,
    check_driver_workflow,
    check_model_update,
    check_residual_update,
)

for validator in (
    check_config,
    check_residual_update,
    check_model_update,
    check_driver_workflow,
):
    print(validator.__name__, inspect.signature(validator))

## Example 1: read-only configuration validation

No data download is needed for any example.


In [ ]:
from astroviper.processing_functions.imaging.config_check import (
    CONTINUUM_SINGLE_FIELD,
    ImagingMode,
    check_config,
)

config = {
    "image_params": {
        "fft_padding": 1.2,
        "frequency_coords": [1.0e9, 1.1e9],
        "reference_frequency": 1.05e9,
        "n_taylor_terms": 2,
    },
    "deconvolver": "mtmfs",
    "continuum_mode": "mfs",
    "cycle_control_params": {"threshold": 0.0},
}
check_config(config, mode=CONTINUUM_SINGLE_FIELD)

Invalid combinations raise without substituting a different algorithm. Both multifield modes are reserved explicitly. At a direct imaging entry point leave `input_checked=False`; only a caller that has validated the unchanged configuration in the same imaging mode should pass `True`. Runtime product checks remain active.

In [ ]:
invalid = {**config, "deconvolver": "hogbom"}
try:
    check_config(invalid, mode=CONTINUUM_SINGLE_FIELD)
except ValueError as error:
    assert "n_taylor_terms=1" in str(error)
else:
    raise AssertionError("Invalid configuration was accepted")
assert invalid["deconvolver"] == "hogbom"

for imaging_kind in ("cube", "continuum"):
    try:
        check_config({}, mode=ImagingMode(imaging_kind, "multifield"))
    except NotImplementedError:
        pass
    else:
        raise AssertionError("Multifield execution is not implemented")

## Coordinate-only spectral preflight

Distributed drivers inspect nominal selected frequencies with equal sample weights.
The column-normalized rank diagnostic runs in float64 independently of image
precision. It reads no bulk weights, flags, UVW, or visibility arrays. Original-basis
conditioning and order-zero noise amplification are equal-weight estimates;
actual positive-weight support and PSF safety require data-dependent checks.


In [ ]:
from astroviper.processing_functions.imaging.config_check.param_check_residual_update import (
    check_spectral_sampling,
)

spectral = check_spectral_sampling(
    [1.0e9, 1.1e9, 1.2e9],
    [1.0, 1.0, 1.0],
    n_taylor_terms=2,
    reference_frequency=1.05e9,
    context={
        "field": "target",
        "spectral_window": "spw0",
        "channel_selection": [0, 1, 2],
        "polarization": ["I"],
    },
)
assert spectral["usable_channel_count"] == 3
assert spectral["spectral_design_rank"] == 2
assert spectral["spectral_design_rank_precision"] == "float64"
spectral

## Optional Gaussian spatial-block diagnostic

Callers that already have spectral weights and projected-baseline estimates can
also examine spatial Fourier blocks using Gaussian channel PSFs. Their FWHM is
wavelength divided by the supplied longest usable projected baseline. This
standalone helper does not collect data, and the coordinate-only distributed
preflight does not invoke it. Its grid is coarse; actual PSFs are checked later
on all modes of their retained paired-lag patch. Each Gaussian channel is
limited to the circular support envelope `|u| <= Bmax * frequency / c`; modes
outside every channel are unsupported, not conditioning failures.


In [ ]:
from astroviper.processing_functions.imaging.config_check.param_check_model_update import (
    check_taylor_hessian_approximation,
)

approximate = check_taylor_hessian_approximation(
    [1.0e9, 1.1e9, 1.2e9],
    [1.0, 1.0, 1.0],
    n_taylor_terms=2,
    reference_frequency=1.1e9,
    longest_projected_baselines=[10000.0, 10000.0, 10000.0],
    image_size=[16, 16],
    cell_size=[-1e-5, 1e-5],
)
assert (
    approximate["spatial_blocks"]["representation"] == "periodic_spatial_fourier_blocks"
)
{
    key: approximate["spatial_blocks"][key]
    for key in (
        "supported_mode_count",
        "unsupported_mode_count",
        "max_order_zero_noise_amplification",
    )
}

## Derived Taylor products

Product validation always runs, including when configuration was checked by an
upper layer. Every time/polarization plane must pass before any CLEAN update.
Here the two-term PSF moments form a positive definite Hessian. The recorded
Taylor-Hessian metric is the native infinity-norm estimate, separate from the
spectral SVD metric.

In [ ]:
import numpy as np
import xarray as xr

from astroviper.processing_functions.imaging.config_check.param_check_model_update import (
    check_model_products,
)

psf = np.zeros((1, 3, 1, 8, 8), dtype=np.float64)
psf[0, 0, 0, 4, 4] = 1.0
psf[0, 2, 0, 4, 4] = 0.25
image = xr.Dataset(
    {
        "SKY_RESIDUAL": (
            ("time", "taylor_term", "polarization", "l", "m"),
            np.ones((1, 2, 1, 8, 8)),
        ),
        "POINT_SPREAD_FUNCTION": (
            ("time", "psf_taylor_order", "polarization", "l", "m"),
            psf,
        ),
    },
    attrs={
        "data_groups": {
            "residual": {
                "sky": "SKY_RESIDUAL",
                "point_spread_function": "POINT_SPREAD_FUNCTION",
            }
        }
    },
)
products = check_model_products(image, config, mode=CONTINUUM_SINGLE_FIELD)
assert products["effective_scales"] == [0.0]
assert products["taylor_hessian_condition_number"]
products

## Spatial coupling invisible at the central pixel

Here the local Taylor Hessian is the identity, but Taylor order one couples
neighbouring pixels. The full convolution operator has stronger coupling at
some spatial frequencies. Both Gaussian preflight and actual-PSF validation use
the same Fourier-block diagnostic. Missing spatial modes are reported rather
than required to be invertible. Rank/curvature problems and amplification above
three on supported modes produce ToolVIPER warning-level log messages; native local inversion guards remain active.

Amplification assumes residual covariance proportional to each Hessian block.
The shift-invariant periodic model excludes finite boundaries and CLEAN masks.
For even actual-PSF axes, index zero is omitted to retain paired lags; the
imaging PSF is unchanged. Self-adjointness uses a spatial FFT roundoff allowance
scaling with `epsilon * ceil(log2(N_pixels)) * sum(abs(PSF_order))`.
Three is a diagnostic policy, not a guarantee about the final residual.


In [ ]:
coupled = image.copy(deep=True)
coupled.POINT_SPREAD_FUNCTION.values[...] = 0
coupled.POINT_SPREAD_FUNCTION.values[0, 0, 0, 4, 4] = 1
coupled.POINT_SPREAD_FUNCTION.values[0, 2, 0, 4, 4] = 1
coupled.POINT_SPREAD_FUNCTION.values[0, 1, 0, 3, 4] = 0.49
coupled.POINT_SPREAD_FUNCTION.values[0, 1, 0, 5, 4] = 0.49
coupled_diagnostics = check_model_products(
    coupled, {"deconvolver": "mtmfs"}, mode=CONTINUUM_SINGLE_FIELD
)
local = coupled_diagnostics["taylor_hessian_order_zero_noise_amplification"][0]
spatial = coupled_diagnostics["taylor_hessian_spatial_blocks"][0]
assert np.isclose(local["per_scale"][0], 1)
assert spatial["max_order_zero_noise_amplification"] > 3
assert spatial["excess_noise_mode_count"] > 0
{
    "local_amplification": local["per_scale"][0],
    "max_spatial_amplification": spatial["max_order_zero_noise_amplification"],
    "worst_mode": spatial["worst_mode"],
}

## Early recipe, geometry, and output checks

The shared validators now reject unsupported FFT/PSF fitters, invalid beam apertures, kernel recipes and padding, mismatched control arrays, and unrepresentable FITS coordinates before execution. Nonuniform FITS channel centres are rejected rather than replaced by a linear approximation; use Zarr to retain those centres.

Kernel preflight reuses the production design cache. Input structure checks inspect dimensions and dtypes without reading visibility samples. Antenna-dependent beam inference runs before the first continuum output write; task-layout checks run again when automatically chosen task extents are known. Static configuration predicates are shared with the parameter module; actual-product and native-buffer guards remain active.


In [ ]:
from astroviper.processing_functions.imaging.config_check import CUBE_SINGLE_FIELD

invalid_examples = [
    (CONTINUUM_SINGLE_FIELD, {"fft_backend": "unknown"}),
    (
        CONTINUUM_SINGLE_FIELD,
        {"continuum_mode": "mvc", "image_params": {"time_coords": [0.0, 1.0]}},
    ),
    (CUBE_SINGLE_FIELD, {"gridding_kernel_params": {"support": 4}}),
    (CUBE_SINGLE_FIELD, {"primary_beam_params": {"effective_dish_diameter": -12.0}}),
]
for mode, candidate in invalid_examples:
    try:
        check_config(candidate, mode=mode)
    except (TypeError, ValueError) as error:
        print(error)
    else:
        raise AssertionError("Invalid configuration was accepted")

## Notes and limitations

- The examples validate partial helper configurations. Distributed applications
  additionally require complete geometry, weighting, and cycle controls.
- Coordinate-only spectral preflight uses equal weights. It does not inspect
  bulk flags, weights, UVW values, or visibilities.
- Gaussian support envelopes are approximations; they do not describe UV holes
  or directional coverage. Actual-product and native inversion guards remain
  necessary even after configuration validation.
- Diagnostic messages use ToolVIPER's logger. Warning-level diagnostics report
  numerical assumptions and do not reject otherwise accepted configurations.
